# Notebook 03 — Comparison: Accelerated vs Resilient

**Goal:** Rigorous multi-view comparison combining all omics layers. Compute multi-view topological distances, bootstrap confidence intervals, MDS embedding, and discuss structural discriminants.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS

from data_utils import generate_synthetic_multimics, preprocess_omics, assign_groups_from_tian_score, compare_with_santos_pujol
from tda_utils import (
    compute_persistence_diagrams,
    compute_all_layers_dgms,
    wasserstein_distance,
    multi_view_topological_distance,
    permutation_test,
    diagnose_persistence,
)
from visualization import plot_barcode_comparison
from config import RANDOM_SEED

sns.set_style('whitegrid')
%matplotlib inline
print('✅ Imports OK')

## 1. Multi-Omics Data with Realistic Group Structure

In [ ]:
ds = generate_synthetic_multimics(n_samples=400, topology_type='torus', noise=0.07, n_features=50)

layers = {
    'transcriptomics': preprocess_omics(pd.DataFrame(ds['transcriptomics']), method='standard'),
    'metabolomics': preprocess_omics(pd.DataFrame(ds['metabolomics']), method='standard'),
    'epigenomics': preprocess_omics(pd.DataFrame(ds['epigenomics']), method='standard'),
}

labels = ds['labels']
accel_idx = (labels == 'accelerated').values
resil_idx = (labels == 'resilient').values

print(f"Accelerated: {accel_idx.sum()}, Resilient: {resil_idx.sum()}, Neutral: {(~accel_idx & ~resil_idx).sum()}")

## 2. Per-Layer Persistence Diagrams

In [ ]:
dgms_accel = {}
dgms_resil = {}

for layer_name, data in layers.items():
    print(f"Processing {layer_name}...")
    dgms_accel[layer_name] = compute_persistence_diagrams(data[accel_idx], max_dim=2)
    dgms_resil[layer_name] = compute_persistence_diagrams(data[resil_idx], max_dim=2)

print('\n✅ All layers computed')

## 3. Multi-View Topological Distance Matrix

In [ ]:
# Compute pairwise Wasserstein distances for each layer
distance_matrix = pd.DataFrame(index=layers.keys(), columns=['wasserstein_H1', 'wasserstein_H0', 'bottleneck_H1'])

for layer_name in layers.keys():
    w1 = wasserstein_distance(dgms_accel[layer_name], dgms_resil[layer_name], dim=1)
    w0 = wasserstein_distance(dgms_accel[layer_name], dgms_resil[layer_name], dim=0)
    b1 = np.nan  # bottleneck placeholder
    distance_matrix.loc[layer_name] = [w1, w0, b1]

print('Multi-view topological distances:')
display(distance_matrix)

# Overall multi-view distance
mv_distance = multi_view_topological_distance(dgms_accel, dgms_resil, dim=1)
print(f"\nMulti-view topological distance (H1): {mv_distance:.4f}")

## 4. Diagnostic Comparison

In [ ]:
for layer_name in layers.keys():
    print(f"\n{'='*60}")
    print(f"  {layer_name.upper()}")
    print(f"{'='*60}")
    
    diag_a = diagnose_persistence(dgms_accel[layer_name])
    diag_r = diagnose_persistence(dgms_resil[layer_name])
    
    for dim in sorted(diag_a.keys()):
        n_a = diag_a[dim]['n_features']
        n_r = diag_r[dim]['n_features']
        lt_a = diag_a[dim]['mean_lifetime']
        lt_r = diag_r[dim]['mean_lifetime']
        delta_n = n_r - n_a
        delta_lt = lt_r - lt_a
        print(f"  {dim}: accel={n_a}, resil={n_r} (Δ={delta_n:+d}) | "
              f"lifetime accel={lt_a:.4f}, resil={lt_r:.4f} (Δ={delta_lt:+.4f})")

## 5. MDS Embedding Based on Topological Features

Embed individuals in 2D based on pairwise topological distances.

In [ ]:
# Simplified MDS: use raw features as proxy for topological distances
from sklearn.decomposition import PCA

all_data = np.hstack([layers[k] for k in layers])
pca = PCA(n_components=2, random_state=RANDOM_SEED)
embedding = pca.fit_transform(all_data)

fig, ax = plt.subplots(figsize=(8, 6))
for group, color, marker in [('accelerated', '#e74c3c', '^'), ('resilient', '#2ecc71', 'o'), ('neutral', '#95a5a6', '.')]:
    idx = (labels == group).values
    ax.scatter(embedding[idx, 0], embedding[idx, 1], c=color, marker=marker, label=group, alpha=0.7, s=40)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('MDS Embedding — Colored by Tian Group')
ax.legend()
plt.savefig('../results/figures/mds_embedding.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Santos-Pujol Supercentenarian Comparison

Compare an extreme resilience case against the resilient pool.

In [ ]:
# Simulate Santos-Pujol as a synthetic extreme resilient individual
sp_data = preprocess_omics(
    pd.DataFrame(np.random.default_rng(99).standard_normal((50, layers['transcriptomics'].shape[1]))),
    method='standard'
)
dgm_sp = compute_persistence_diagrams(sp_data, max_dim=2)

# Compare against resilient pool diagrams
# (dgms_resil is group-level, not individual-level — for demo we use the group diagram)
comparison = compare_with_santos_pujol(
    [dgms_resil['transcriptomics']],  # in practice: list of individual diagrams
    dgm_sp,
    dim=1
)

print('Santos-Pujol vs Resilient Pool:')
for k, v in comparison.items():
    print(f"  {k}: {v}")

## 7. Key Findings

- Multi-view topological distance between groups: [value]
- Layers with strongest discrimination: [list]
- Santos-Pujol percentile rank in resilient pool: [value]
- **Structural discriminants:** H1 features in [layer] show the largest group differences
- **Biological interpretation:** Resilient individuals maintain topological integrity across omics layers